# Automotive Knowledge Graph - Tutorial Interactivo

Este notebook demuestra cómo trabajar con el grafo de conocimiento automotriz usando RDFLib y SPARQLWrapper.

## Objetivos
1. Cargar la ontología y datos RDF
2. Ejecutar consultas SPARQL locales
3. Consultar el endpoint Fuseki remoto
4. Analizar resultados con Pandas
5. Visualizar el grafo con NetworkX

## 1. Setup y Dependencias

In [ ]:
# Instalar dependencias (ejecutar solo una vez)
# !pip install rdflib SPARQLWrapper pandas matplotlib networkx

In [ ]:
import rdflib
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef, Literal
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict

print(f"✅ RDFLib version: {rdflib.__version__}")

## 2. Cargar Ontología y Datos

In [ ]:
# Crear grafo RDF
g = Graph()

# Cargar ontología
print("📚 Cargando ontología...")
g.parse("/home/jovyan/ontology/automotive-ontology.ttl", format="turtle")
ontology_triples = len(g)
print(f"   {ontology_triples} tripletas de ontología cargadas")

# Cargar datos de instancias
print("📦 Cargando datos de instancias...")
g.parse("/home/jovyan/data/automotive-instances.ttl", format="turtle")
total_triples = len(g)
instance_triples = total_triples - ontology_triples
print(f"   {instance_triples} tripletas de instancias cargadas")

print(f"\n✅ Total: {total_triples} tripletas en memoria")

## 3. Definir Namespaces

In [ ]:
# Definir namespaces
AUTO = Namespace("http://example.org/automotive#")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

# Bind para serialización
g.bind("auto", AUTO)
g.bind("xsd", XSD)

print("✅ Namespaces configurados")

## 4. Consultas Básicas con RDFLib

In [ ]:
# Listar todas las clases OWL
print("🔍 Clases OWL definidas:\n")

classes = []
for s in g.subjects(RDF.type, OWL.Class):
    label = g.value(s, RDFS.label, default=str(s).split('#')[-1])
    classes.append(str(label))

classes.sort()
for i, cls in enumerate(classes[:10], 1):
    print(f"  {i}. {cls}")

print(f"\n... y {len(classes) - 10} clases más")

In [ ]:
# Listar todos los vehículos eléctricos
print("🚗 Vehículos Eléctricos:\n")

electric_vehicles = []
for s in g.subjects(RDF.type, AUTO.VehiculoElectrico):
    name = g.value(s, AUTO.nombreComercial)
    autonomy = g.value(s, AUTO.autonomiaKm)
    battery = g.value(s, AUTO.capacidadBateriaKWh)
    
    if name:
        electric_vehicles.append({
            'Modelo': str(name),
            'Autonomía (km)': float(autonomy) if autonomy else None,
            'Batería (kWh)': float(battery) if battery else None
        })

# Convertir a DataFrame
df_evs = pd.DataFrame(electric_vehicles)
df_evs = df_evs.sort_values('Autonomía (km)', ascending=False)
df_evs

## 5. Consultas SPARQL Locales

In [ ]:
# Consulta SPARQL: Marcas por grupo corporativo
query = """
PREFIX auto: <http://example.org/automotive#>

SELECT ?grupo ?marca ?pais
WHERE {
  ?marcaUri auto:perteneceA ?grupoUri ;
            auto:nombreComercial ?marca .
  ?grupoUri auto:nombreComercial ?grupo .
  OPTIONAL { ?marcaUri auto:paisOrigen ?pais }
}
ORDER BY ?grupo ?marca
"""

results = g.query(query)

print("🏢 Marcas por Grupo Corporativo:\n")
data = []
for row in results:
    data.append({
        'Grupo': str(row.grupo),
        'Marca': str(row.marca),
        'País': str(row.pais) if row.pais else 'N/A'
    })

df_brands = pd.DataFrame(data)
df_brands

In [ ]:
# Consulta SPARQL: Vehículos con más de 300 HP
query = """
PREFIX auto: <http://example.org/automotive#>

SELECT ?nombre ?fabricante ?potencia ?aceleracion
WHERE {
  ?vehiculo auto:nombreComercial ?nombre ;
            auto:potenciaHP ?potencia ;
            auto:fabricadoPor ?fabUri .
  ?fabUri auto:nombreComercial ?fabricante .
  OPTIONAL { ?vehiculo auto:aceleracion0a100 ?aceleracion }
  FILTER(?potencia > 300)
}
ORDER BY DESC(?potencia)
"""

results = g.query(query)

print("⚡ Vehículos de Alto Rendimiento (>300 HP):\n")
data = []
for row in results:
    data.append({
        'Modelo': str(row.nombre),
        'Fabricante': str(row.fabricante),
        'Potencia (HP)': float(row.potencia),
        '0-100 km/h (s)': float(row.aceleracion) if row.aceleracion else None
    })

df_performance = pd.DataFrame(data)
df_performance

## 6. Consultas al Endpoint Fuseki Remoto

In [ ]:
# Configurar SPARQLWrapper para Fuseki
sparql = SPARQLWrapper("http://fuseki:3030/automotive/query")
sparql.setReturnFormat(JSON)

print("✅ Conectado a Fuseki endpoint")

In [ ]:
# Consulta remota con inferencia OWL
query = """
PREFIX auto: <http://example.org/automotive#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?plataforma (COUNT(?vehiculo) as ?numVehiculos)
WHERE {
  ?plataforma a auto:PlataformaTecnica ;
              auto:nombreComercial ?nombrePlataforma .
  ?vehiculo auto:compartePlataforma ?plataforma .
}
GROUP BY ?plataforma ?nombrePlataforma
HAVING (COUNT(?vehiculo) > 1)
ORDER BY DESC(?numVehiculos)
"""

sparql.setQuery(query)
results = sparql.query().convert()

print("🔧 Plataformas Técnicas Compartidas:\n")
for result in results["results"]["bindings"]:
    plataforma = result["plataforma"]["value"].split('#')[-1]
    num = result["numVehiculos"]["value"]
    print(f"  {plataforma}: {num} vehículos")

## 7. Análisis de Datos con Pandas

In [ ]:
# Extraer todos los vehículos con sus especificaciones
query = """
PREFIX auto: <http://example.org/automotive#>

SELECT ?nombre ?año ?potencia ?peso ?precio
WHERE {
  ?vehiculo a auto:Vehiculo ;
            auto:nombreComercial ?nombre ;
            auto:añoLanzamiento ?año .
  OPTIONAL { ?vehiculo auto:potenciaHP ?potencia }
  OPTIONAL { ?vehiculo auto:pesoKg ?peso }
  OPTIONAL { ?vehiculo auto:precioBase ?precio }
}
"""

results = g.query(query)

data = []
for row in results:
    data.append({
        'Modelo': str(row.nombre),
        'Año': int(row.año),
        'Potencia (HP)': float(row.potencia) if row.potencia else None,
        'Peso (kg)': float(row.peso) if row.peso else None,
        'Precio (USD)': float(row.precio) if row.precio else None
    })

df_vehicles = pd.DataFrame(data)

# Calcular relación peso/potencia
df_vehicles['Peso/Potencia'] = df_vehicles['Peso (kg)'] / df_vehicles['Potencia (HP)']

print("📊 Estadísticas de Vehículos:\n")
print(df_vehicles.describe())

In [ ]:
# Visualización: Potencia vs Precio
fig, ax = plt.subplots(figsize=(10, 6))

df_plot = df_vehicles.dropna(subset=['Potencia (HP)', 'Precio (USD)'])

ax.scatter(df_plot['Potencia (HP)'], df_plot['Precio (USD)'], 
           s=100, alpha=0.6, c='steelblue', edgecolors='black')

# Anotar algunos puntos
for idx, row in df_plot.iterrows():
    if row['Potencia (HP)'] > 300 or row['Precio (USD)'] > 60000:
        ax.annotate(row['Modelo'][:20], 
                   (row['Potencia (HP)'], row['Precio (USD)']),
                   fontsize=8, alpha=0.7, 
                   xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Potencia (HP)', fontsize=12)
ax.set_ylabel('Precio Base (USD)', fontsize=12)
ax.set_title('Relación Potencia vs Precio - Mercado Automotriz', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Visualización del Grafo de Conocimiento

In [ ]:
# Crear un subgrafo de relaciones corporativas
G = nx.DiGraph()

# Añadir relaciones grupo -> marca
query = """
PREFIX auto: <http://example.org/automotive#>

SELECT ?grupo ?marca
WHERE {
  ?marcaUri auto:perteneceA ?grupoUri ;
            auto:nombreComercial ?marca .
  ?grupoUri auto:nombreComercial ?grupo .
}
"""

results = g.query(query)

for row in results:
    G.add_edge(str(row.grupo), str(row.marca), relation="posee")

# Visualizar
fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Nodos
grupos = [n for n in G.nodes() if G.in_degree(n) == 0]  # Sin padres = grupos
marcas = [n for n in G.nodes() if G.in_degree(n) > 0]   # Con padres = marcas

nx.draw_networkx_nodes(G, pos, nodelist=grupos, 
                       node_color='orange', node_size=2000, 
                       node_shape='s', alpha=0.9, ax=ax)

nx.draw_networkx_nodes(G, pos, nodelist=marcas, 
                       node_color='lightblue', node_size=1200, 
                       alpha=0.8, ax=ax)

# Aristas
nx.draw_networkx_edges(G, pos, edge_color='gray', 
                       arrows=True, arrowsize=20, 
                       arrowstyle='->', alpha=0.6, ax=ax)

# Labels
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

ax.set_title('Grafo de Conocimiento - Jerarquía Corporativa Automotriz', 
             fontsize=16, fontweight='bold')
ax.axis('off')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='orange', label='Grupo Corporativo'),
    Patch(facecolor='lightblue', label='Marca')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Análisis de Propulsión por Fabricante

In [ ]:
# Consulta: Tipos de propulsión por fabricante
query = """
PREFIX auto: <http://example.org/automotive#>

SELECT ?fabricante ?tipoPropulsion (COUNT(?vehiculo) as ?cantidad)
WHERE {
  ?vehiculo auto:fabricadoPor ?fabUri ;
            auto:tienePropulsion ?tipoUri .
  ?fabUri auto:nombreComercial ?fabricante .
  ?tipoUri rdfs:label ?tipoPropulsion .
  FILTER(LANG(?tipoPropulsion) = "es" || LANG(?tipoPropulsion) = "")
}
GROUP BY ?fabricante ?tipoPropulsion
ORDER BY ?fabricante
"""

results = g.query(query)

# Organizar datos
propulsion_data = defaultdict(lambda: defaultdict(int))
for row in results:
    fab = str(row.fabricante)
    tipo = str(row.tipoPropulsion)
    cantidad = int(row.cantidad)
    propulsion_data[fab][tipo] = cantidad

# Crear DataFrame
df_propulsion = pd.DataFrame(propulsion_data).T.fillna(0)

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
df_propulsion.plot(kind='bar', stacked=True, ax=ax, 
                   color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])

ax.set_xlabel('Fabricante', fontsize=12)
ax.set_ylabel('Número de Modelos', fontsize=12)
ax.set_title('Distribución de Tipos de Propulsión por Fabricante', 
             fontsize=14, fontweight='bold')
ax.legend(title='Tipo de Propulsión', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Exportar Resultados

In [ ]:
# Exportar vehículos eléctricos a CSV
df_evs.to_csv('/home/jovyan/work/electric_vehicles.csv', index=False)
print("✅ Exportado: electric_vehicles.csv")

# Exportar especificaciones completas
df_vehicles.to_csv('/home/jovyan/work/all_vehicles.csv', index=False)
print("✅ Exportado: all_vehicles.csv")

# Exportar un subgrafo a formato RDF
subgraph = Graph()
for s, p, o in g.triples((None, AUTO.fabricadoPor, None)):
    subgraph.add((s, p, o))
    # Añadir nombre del vehículo
    name = g.value(s, AUTO.nombreComercial)
    if name:
        subgraph.add((s, AUTO.nombreComercial, name))

subgraph.serialize('/home/jovyan/work/vehicles_manufacturers.ttl', format='turtle')
print("✅ Exportado: vehicles_manufacturers.ttl")

print(f"\n📁 Archivos guardados en /home/jovyan/work/")

## 🎯 Conclusiones

Este notebook ha demostrado:

1. ✅ Carga de ontologías OWL/RDFS en RDFLib
2. ✅ Consultas SPARQL locales y remotas (Fuseki)
3. ✅ Análisis de datos semánticos con Pandas
4. ✅ Visualización de grafos de conocimiento con NetworkX
5. ✅ Exportación de resultados en múltiples formatos

### Próximos Pasos

- Añadir más instancias de vehículos
- Explorar consultas federadas (SPARQL 1.1 Federation)
- Implementar razonamiento personalizado con reglas SWRL
- Integrar con APIs externas (precios de mercado, reviews)
- Crear dashboard interactivo con Plotly/Dash